<a href="https://colab.research.google.com/github/Lo0X/Lo0X/blob/Atv_Lucca/IMAGEM_MEME_KEKW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# PROJETO - PROCESSAMENTO DE IMAGENS / MEME KEKW
# Google Colab + OpenCV
# ============================================================

import cv2 # Biblioteca para matrizes em Numpy
import numpy as np # Armazena a imagem como uma Matriz
from google.colab import files # Trbalhar com arquivos
from google.colab.patches import cv2_imshow # Visualizar a imagem dentro do Colab :)


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def mostrar_imagem(imagem, titulo="Imagem atual"):  # Função para armazenar e mostrar imagem que será carregada
    print(f"\n--- {titulo} ---")
    cv2_imshow(imagem)


def dados_imagem(imagem):  # Função para exibir os detalhes da imagem carregada.
    """Exibe resolução, quantidade de pixels e canais."""

    altura, largura = imagem.shape[:2]  # Informa o formato da nossa matriz, altura, largura e canais
    total_pixels = largura * altura  # Calcula os pixels Altura * Largrua

    # Imagem em escala de cinza possui apenas 2 dimensões
    if len(imagem.shape) == 2:
        canais = 1
    else:
        canais = imagem.shape[2]

    print("\n========== DADOS DA IMAGEM ==========")
    print(f"Largura: {largura} pixels")
    print(f"Altura: {altura} pixels")
    print(f"Resolução: {largura} x {altura}")
    print(f"Total de pixels: {total_pixels}")
    print(f"Número de canais: {canais}")
    print("=====================================")


def converter_cinza(imagem):
    """Converte imagem colorida BGR para tons de cinza."""

    if len(imagem.shape) == 2:
        print("\nA imagem já está em tons de cinza.")
        return imagem

    return cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)


def ajustar_brilho_contraste(imagem, ganho, bias):
    """
    Aplica:
    Pnovo = Patual * Ganho + Bias

    O np.clip garante que os valores continuem entre 0 e 255.
    """

    imagem_float = imagem.astype(np.float32)

    resultado = imagem_float * ganho + bias

    resultado = np.clip(resultado, 0, 255)

    return resultado.astype(np.uint8)


def rotacionar_imagem(imagem, angulo):
    """Rotaciona a imagem em torno do seu centro."""

    altura, largura = imagem.shape[:2]

    centro = (largura / 2, altura / 2)

    matriz_rotacao = cv2.getRotationMatrix2D(
        centro,
        angulo,
        1.0
    )

    resultado = cv2.warpAffine(
        imagem,
        matriz_rotacao,
        (largura, altura)
    )

    return resultado


def espelhar_imagem(imagem, tipo):
    """Espelhamento horizontal ou vertical."""

    if tipo == "H":
        # flipCode = 1 -> horizontal
        return cv2.flip(imagem, 1)

    elif tipo == "V":
        # flipCode = 0 -> vertical
        return cv2.flip(imagem, 0)

    else:
        print("Opção inválida.")
        return imagem


def efeito_pixel_art(imagem, fator=8, niveis=4):
    """
    Cria efeito Pixel Art através de:

    1. Redução da resolução
    2. Ampliação usando Nearest Neighbor
    3. Quantização das cores
    """

    altura, largura = imagem.shape[:2]

    # Evita dimensões iguais a zero
    nova_largura = max(1, largura // fator)
    nova_altura = max(1, altura // fator)

    # -------------------------
    # Redução da imagem
    # -------------------------

    pequena = cv2.resize(
        imagem,
        (nova_largura, nova_altura),
        interpolation=cv2.INTER_NEAREST
    )

    # -------------------------
    # Ampliação
    # -------------------------

    pixelada = cv2.resize(
        pequena,
        (largura, altura),
        interpolation=cv2.INTER_NEAREST
    )

    # -------------------------
    # Quantização
    # -------------------------

    passo = 256 // niveis

    quantizada = (pixelada // passo) * passo

    return quantizada.astype(np.uint8)


def filtro_cor(imagem, cor):
    """
    Mantém somente o canal escolhido.

    OpenCV trabalha no padrão BGR:
    B = Azul
    G = Verde
    R = Vermelho
    """

    # Se estiver em escala de cinza,
    # convertemos novamente para BGR.
    if len(imagem.shape) == 2:
        imagem = cv2.cvtColor(imagem, cv2.COLOR_GRAY2BGR)

    resultado = np.zeros_like(imagem)

    if cor == "V":
        # Canal vermelho = índice 2
        resultado[:, :, 2] = imagem[:, :, 2]

    elif cor == "G":
        # Canal verde = índice 1
        resultado[:, :, 1] = imagem[:, :, 1]

    elif cor == "A":
        # Canal azul = índice 0
        resultado[:, :, 0] = imagem[:, :, 0]

    else:
        print("Cor inválida.")
        return imagem

    return resultado


# ============================================================
# UPLOAD DA IMAGEM
# ============================================================

print("==========================================")
print("   PROCESSADOR DE IMAGENS - MEME")
print("==========================================")

print("\nFaça o upload de uma imagem:")

arquivos = files.upload()

if len(arquivos) == 0:
    raise Exception("Nenhuma imagem foi enviada.")

nome_arquivo = list(arquivos.keys())[0]

imagem_original = cv2.imread(nome_arquivo)

if imagem_original is None:
    raise Exception("Não foi possível abrir a imagem.")

# A imagem atual começa como cópia da original
imagem_atual = imagem_original.copy()

# Histórico para permitir DESFAZER
historico = []

print("\nImagem carregada com sucesso!")

mostrar_imagem(imagem_atual, "Imagem Original")


# ============================================================
# MENU PRINCIPAL
# ============================================================

while True:

    print("\n")
    print("==========================================")
    print("             MENU PRINCIPAL")
    print("==========================================")
    print("[ 1 ] Exibir Dados da Imagem")
    print("[ 2 ] Converter para Tons de Cinza")
    print("[ 3 ] Ajustar Brilho e Contraste")
    print("[ 4 ] Rotacionar Imagem")
    print("[ 5 ] Espelhar Imagem")
    print("[ 6 ] Efeito Pixel Art")
    print("[ 7 ] Aplicar Filtro de Cor")
    print("[ 8 ] Desfazer Última Ação")
    print("[ 9 ] Salvar e Sair")
    print("==========================================")

    opcao = input("Escolha uma opção: ").strip()

    # ========================================================
    # OPÇÃO 1
    # ========================================================

    if opcao == "1":

        dados_imagem(imagem_atual)

        mostrar_imagem(
            imagem_atual,
            "Imagem Atual"
        )

    # ========================================================
    # OPÇÃO 2
    # ========================================================

    elif opcao == "2":

        # Guarda o estado ANTES da alteração
        historico.append(imagem_atual.copy())

        imagem_atual = converter_cinza(imagem_atual)

        print("\nConversão para tons de cinza realizada!")

        mostrar_imagem(
            imagem_atual,
            "Tons de Cinza"
        )

    # ========================================================
    # OPÇÃO 3
    # ========================================================

    elif opcao == "3":

        try:

            ganho = float(
                input(
                    "\nDigite o GANHO para contraste "
                    "(ex: 1.2): "
                )
            )

            bias = float(
                input(
                    "Digite o BIAS para brilho "
                    "(ex: 30): "
                )
            )

            historico.append(imagem_atual.copy())

            imagem_atual = ajustar_brilho_contraste(
                imagem_atual,
                ganho,
                bias
            )

            print("\nBrilho e contraste ajustados!")

            mostrar_imagem(
                imagem_atual,
                "Brilho e Contraste"
            )

        except ValueError:

            print("\nDigite valores numéricos válidos.")

    # ========================================================
    # OPÇÃO 4
    # ========================================================

    elif opcao == "4":

        try:

            angulo = float(
                input(
                    "\nDigite o ângulo da rotação "
                    "(em graus): "
                )
            )

            historico.append(imagem_atual.copy())

            imagem_atual = rotacionar_imagem(
                imagem_atual,
                angulo
            )

            print("\nRotação realizada!")

            mostrar_imagem(
                imagem_atual,
                f"Rotação de {angulo} graus"
            )

        except ValueError:

            print("\nDigite um ângulo válido.")

    # ========================================================
    # OPÇÃO 5
    # ========================================================

    elif opcao == "5":

        print("\nH - Espelhamento Horizontal")
        print("V - Espelhamento Vertical")

        tipo = input(
            "Escolha H ou V: "
        ).strip().upper()

        if tipo in ["H", "V"]:

            historico.append(imagem_atual.copy())

            imagem_atual = espelhar_imagem(
                imagem_atual,
                tipo
            )

            print("\nImagem espelhada!")

            mostrar_imagem(
                imagem_atual,
                "Imagem Espelhada"
            )

        else:

            print("\nOpção inválida.")

    # ========================================================
    # OPÇÃO 6
    # ========================================================

    elif opcao == "6":

        try:

            fator = int(
                input(
                    "\nDigite o fator de pixelização "
                    "(recomendado: 4, 8 ou 16): "
                )
            )

            niveis = int(
                input(
                    "Digite a quantidade de níveis de cor "
                    "(recomendado: 4 ou 8): "
                )
            )

            if fator <= 0 or niveis <= 0 or niveis > 256:
                print("\nValores inválidos.")
                continue

            historico.append(imagem_atual.copy())

            imagem_atual = efeito_pixel_art(
                imagem_atual,
                fator,
                niveis
            )

            print("\nEfeito Pixel Art aplicado!")

            mostrar_imagem(
                imagem_atual,
                "Pixel Art"
            )

        except ValueError:

            print("\nDigite números inteiros válidos.")

    # ========================================================
    # OPÇÃO 7
    # ========================================================

    elif opcao == "7":

        print("\nEscolha a cor que deseja manter:")
        print("V - Vermelho")
        print("G - Verde")
        print("A - Azul")

        cor = input(
            "Escolha V, G ou A: "
        ).strip().upper()

        if cor in ["V", "G", "A"]:

            historico.append(imagem_atual.copy())

            imagem_atual = filtro_cor(
                imagem_atual,
                cor
            )

            print("\nFiltro aplicado!")

            mostrar_imagem(
                imagem_atual,
                "Filtro de Cor"
            )

        else:

            print("\nCor inválida.")

    # ========================================================
    # OPÇÃO 8 - DESFAZER
    # ========================================================

    elif opcao == "8":

        if len(historico) > 0:

            # Recupera o último estado salvo
            imagem_atual = historico.pop()

            print("\nÚltima operação desfeita!")

            mostrar_imagem(
                imagem_atual,
                "Estado Anterior"
            )

        else:

            print(
                "\nNão existe nenhuma operação para desfazer."
            )

    # ========================================================
    # OPÇÃO 9 - SALVAR
    # ========================================================

    elif opcao == "9":

        nome_saida = input(
            "\nDigite o nome do arquivo "
            "(ex: resultado.jpg): "
        ).strip()

        # Se o usuário esquecer a extensão
        if "." not in nome_saida:
            nome_saida += ".jpg"

        sucesso = cv2.imwrite(
            nome_saida,
            imagem_atual
        )

        if sucesso:

            print("\n==========================================")
            print("Imagem salva com sucesso!")
            print(f"Arquivo: {nome_saida}")
            print("==========================================")

            # Faz o download automaticamente no Colab
            files.download(nome_saida)

        else:

            print("\nErro ao salvar a imagem.")

        break

    # ========================================================
    # OPÇÃO INVÁLIDA
    # ========================================================

    else:

        print(
            "\nOpção inválida! "
            "Escolha uma opção entre 1 e 9."
        )


print("\nPrograma encerrado.")